# 01 - Eksplorasi penjualan retail
Data sintetis, 100.000 transaksi, 12 kolom sumber. Tujuan: tren bulanan, kontribusi cabang/kategori, serta preferensi pembayaran. SQL sumber memakai SQLite; PostgreSQL 15 diuji pada notebook 03.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd() if (Path.cwd()/"sales_project.py").exists() else Path.cwd().parent
assert (ROOT/"sales_project.py").exists(), "Buka notebook dari folder proyek/notebooks"
sys.path.insert(0,str(ROOT))
os.chdir(ROOT)
java_candidates = list((ROOT/".runtime/java").glob("*/bin/java.exe"))
if java_candidates:
    os.environ["JAVA_HOME"] = str(java_candidates[0].parent.parent)
os.environ["PYSPARK_PYTHON"] = sys.executable
from sales_project import *
print("Project:", ROOT.name)


Project: big-data-sales-analytics


In [2]:
df, profile = exploration()
display(df.head())
display(profile)
display(df.describe(include="all").T)

          column   dtype  nulls  distinct
  transaction_id   int64      0    100000
transaction_date  object      0      1004
          branch  object      0         8
        category  object      0         6
    product_name  object      0        30
        quantity   int64      0        10
      unit_price float64      0       349
   discount_rate float64      0         5
     gross_sales float64      0      1771
       net_sales float64      0      6097
  payment_method  object      0         4
customer_segment  object      0         3
Pandas checks: {'rows_100000': True, 'at_least_10_columns': True, 'unique_ids': True, 'no_nulls': True, 'valid_dates': True, 'nonempty_dimensions': True, 'finite_numbers': True, 'positive_integer_quantity': True, 'nonnegative_price': True, 'discount_range': True, 'gross_formula': True, 'net_formula': True}


,transaction_id,transaction_date,branch,category,product_name,quantity,unit_price,discount_rate,gross_sales,net_sales,payment_method,customer_segment
0,1,2026-04-13,Denpasar,Kebutuhan Rumah,Botol Minum,2,104000.0,0.15,208000.0,176800.0,Transfer,Retail
1,2,2026-04-10,Surabaya,Olahraga,Raket,6,138000.0,0.05,828000.0,786600.0,Kartu,Retail
2,3,2025-07-06,Bandung,Olahraga,Bola,2,92000.0,0.20,184000.0,147200.0,E-Wallet,Retail
3,4,2025-05-24,Semarang,Olahraga,Bola,7,158000.0,0.05,1106000.0,1050700.0,Tunai,Retail
4,5,2026-05-09,Jakarta,Fashion,Jaket,4,170000.0,0.00,680000.0,680000.0,E-Wallet,Corporate


,column,dtype,nulls,distinct
0,transaction_id,int64,0,100000
1,transaction_date,object,0,1004
2,branch,object,0,8
3,category,object,0,6
4,product_name,object,0,30
5,quantity,int64,0,10
6,unit_price,float64,0,349
7,discount_rate,float64,0,5
8,gross_sales,float64,0,1771
9,net_sales,float64,0,6097


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
transaction_id,100000.0,NaN,NaN,NaN,50000.5,28867.657797,1.0,25000.75,50000.5,75000.25,100000.0
transaction_date,100000,1004,2025-04-15,129,NaN,NaN,NaN,NaN,NaN,NaN,NaN
branch,100000,8,Surabaya,12659,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,100000,6,Kebutuhan Rumah,20122,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name,100000,30,Wadah Makan,4079,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,100000.0,NaN,NaN,NaN,5.51507,2.87536,1.0,3.0,6.0,8.0,10.0
unit_price,100000.0,NaN,NaN,NaN,105403.73,74551.173236,12000.0,51000.0,84000.0,138000.0,360000.0
discount_rate,100000.0,NaN,NaN,NaN,0.0553,0.061197,0.0,0.0,0.05,0.1,0.2
gross_sales,100000.0,NaN,NaN,NaN,580742.26,553599.114567,12000.0,196000.0,405000.0,770000.0,3600000.0
net_sales,100000.0,NaN,NaN,NaN,548566.919,525260.948642,10400.0,184300.0,380000.0,728000.0,3600000.0


## Kebijakan kualitas data
Tanggal dan angka distandarkan. Kolom wajib kosong, angka tidak valid, diskon di luar 0-1, formula meleset > Rp0,02, serta semua baris dengan ID ganda dikarantina. Tidak ada imputasi nilai transaksi tanpa bukti. Dataset utama sudah bersih; uji data rusak terpisah menguji kebijakan ini.

In [3]:
import pandas as pd
display(pd.read_csv(OUT/"profiling/sql_source_monthly.csv"))

,month,transactions,units_sold,net_sales
0,2024-01,3163,17365,1.744161e+09
1,2024-02,2906,15879,1.550383e+09
2,2024-03,3206,17874,1.775595e+09
3,2024-04,2928,16016,1.597627e+09
4,2024-05,3123,17162,1.733182e+09
5,2024-06,2941,15833,1.573383e+09
6,2024-07,3056,16944,1.699022e+09
7,2024-08,3082,17115,1.686428e+09
8,2024-09,3044,16847,1.695288e+09
9,2024-10,3158,17742,1.783478e+09
